In this notebook, we replicate this analysis for the AR6 IMPs.

In [1]:
import multiprocessing
import os
import platform
from functools import partial
from pathlib import Path
import dotenv

import pyam
import numpy as np
import openscm_units
import pandas as pd
import pandas_indexing as pix
import pandas_openscm
import pint
import seaborn as sns

from gcages.ar6 import (
    AR6Harmoniser,
    AR6Infiller,
    AR6PostProcessor,
    AR6PreProcessor,
    AR6SCMRunner,
    get_ar6_full_historical_emissions,
)

from utils import (
    calculate_crossing_timings, 
    extend_flatline_after_netzero, 
    sanitize_label
)

/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
[WARNING] 11:07:55 - pint.util: Redefining 'C' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'N' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'NOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'gNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'tNOX' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'S' (<class 'pint.delegates.txt_defparser.plain.UnitDefinition'>)
[WARNING] 11:07:55 - pint.util: Redefining 'yr' (<cl

In [2]:
# Setup pint
pint.set_application_registry(openscm_units.unit_registry)

In [3]:
pandas_openscm.register_pandas_accessors()

In [4]:
dotenv.load_dotenv()

True

In [5]:
ar6_data = pyam.IamDataFrame(
    Path(
        os.environ['AR6_DATA']
    ) / "AR6_Scenarios_Database_World_v1.1.csv" 
)

[INFO] 11:07:55 - pyam.core: Reading file /Users/ganti/Documents/data/ar6-data/1668008312256-AR6_Scenarios_Database_World_v1.1.csv/AR6_Scenarios_Database_World_v1.1.csv


In [6]:
ar6_data.load_meta(
    Path(
        os.environ['AR6_DATA']
    ) / "AR6_Scenarios_Database_metadata_indicators_v1.1.xlsx"
)

[WARNING] 11:08:03 - pyam.core: Ignoring meta indicators for the following scenarios:
       model              scenario
0  Reference                  CEDS
1  Reference             EDGAR AR6
2  Reference           EMBERS 2020
3  Reference              IEA 2019
4  Reference  Solar-Wind-composite
...


In [7]:
ar6_imp = (
    ar6_data
    .filter(
        IMP_marker='non-IMP',
        keep=False
    )
)

In [8]:
ar6_imp_emissions = (
    ar6_imp
    .filter(
        variable='Emissions*'
    )
)

First, we harmonise the data and infill it.

In [9]:
pre_processor = AR6PreProcessor.from_ar6_config(
    n_processes=10
)

In [10]:
pre_processed = pre_processor(
    ar6_imp_emissions.timeseries()
)

9it [00:00, 149.28it/s]
For each model-scenario, calculating conditional sums: 100%|██████████| 9/9 [00:00<00:00, 25.00it/s]
9it [00:00, 86.61it/s]
For each model-scenario, reclassifying variables: 100%|██████████| 9/9 [00:00<00:00, 23.51it/s]
9it [00:00, 90.76it/s]
For each model-scenario, conditionally removing variables: 100%|██████████| 9/9 [00:00<00:00, 23.77it/s]
9it [00:00, 78.28it/s]
For each model-scenario, dropping variables if they are identical: 100%|██████████| 9/9 [00:00<00:00, 26.05it/s]


In [11]:
AR6_HISTORICAL_EMISSIONS_FILE = Path(
    "../../gcages/tests/regression/ar6/ar6-workflow-inputs/history_ar6.csv"
)

In [12]:
harmoniser = AR6Harmoniser.from_ar6_config(
    ar6_historical_emissions_file=AR6_HISTORICAL_EMISSIONS_FILE,
    n_processes=10
)

In [13]:
harmonised = harmoniser(pre_processed)
harmonised

9it [00:00, 120.50it/s]
  0%|          | 0/9 [00:00<?, ?it/s]INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with reduce_ratio_2150_cov
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with constant_ratio
 11%|█         | 1/9 [00:01<00:09,  1.22s/it]INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_ratio_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2150_cov
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:Harmonizing with constant_ratio
INFO:root:Harmonizing with reduce_offset_2150_cov
INFO:root:Harmonizing with reduce_ratio_2080
INFO:root:

2015  \
model     scenario               region variable                unit                      
IMAGE 3.0 EN_INDCi2030_3000f     World  Emissions|BC            Mt BC/yr       9.727424   
                                        Emissions|CH4           Mt CH4/yr    388.072796   
                                        Emissions|CO            Mt CO/yr     934.349885   
                                        Emissions|CO2|Biosphere Mt CO2/yr   3517.440000   
                                        Emissions|CO2|Fossil    Mt CO2/yr  35635.286300   
...                                                                                 ...   
GCAM 5.3  NGFS2_Current Policies World  Emissions|NOx           Mt NO2/yr    155.520450   
                                        Emissions|OC            Mt OC/yr      34.746024   
                                        Emissions|SF6           kt SF6/yr      8.020000   
                                        Emissions|SOx           Mt SO2/yr    100.771167   
                                        Emissions|NMVOC         Mt VOC/yr    227.244819   

                                                                                   2016  \
model     scenario               region variable                unit                      
IMAGE 3.0 EN_INDCi2030_3000f     World  Emissions|BC            Mt BC/yr       9.718991   
                                        Emissions|CH4           Mt CH4/yr    386.607263   
                                        Emissions|CO            Mt CO/yr     948.512512   
                                        Emissions|CO2|Biosphere Mt CO2/yr   3831.159324   
                                        Emissions|CO2|Fossil    Mt CO2/yr  36006.593585   
...                                                                                 ...   
GCAM 5.3  NGFS2_Current Policies World  Emissions|NOx           Mt NO2/yr    152.211398   
                                        Emissions|OC            Mt OC/yr      33.647810   
                                        Emissions|SF6           kt SF6/yr      8.154513   
                                        Emissions|SOx           Mt SO2/yr     98.986454   
                                        Emissions|NMVOC         Mt VOC/yr    223.367135   

                                                                                   2017  \
model     scenario               region variable                unit                      
IMAGE 3.0 EN_INDCi2030_3000f     World  Emissions|BC            Mt BC/yr       9.710393   
                                        Emissions|CH4           Mt CH4/yr    385.143926   
                                        Emissions|CO            Mt CO/yr     962.644719   
                                        Emissions|CO2|Biosphere Mt CO2/yr   4144.878648   
                                        Emissions|CO2|Fossil    Mt CO2/yr  36377.802621   
...                                                                                 ...   
GCAM 5.3  NGFS2_Current Policies World  Emissions|NOx           Mt NO2/yr    148.912323   
                                        Emissions|OC            Mt OC/yr      32.547851   
                                        Emissions|SF6           kt SF6/yr      8.289026   
                                        Emissions|SOx           Mt SO2/yr     97.168599   
                                        Emissions|NMVOC         Mt VOC/yr    219.502067   

                                                                                   2018  \
model     scenario               region variable                unit                      
IMAGE 3.0 EN_INDCi2030_3000f     World  Emissions|BC            Mt BC/yr       9.701631   
                                        Emissions|CH4           Mt CH4/yr    383.682784   
                                        Emissions|CO            Mt CO/yr     976.746505   
                                        Emissions|CO2|Biosphere Mt CO2/yr   

Next, we infill the emissions

In [14]:
AR6_INFILLING_DB_FILE = Path(
    "../../gcages/tests/regression/ar6/ar6-workflow-inputs/infilling_db_ar6.csv"
)
AR6_INFILLING_DB_CFCS_FILE = Path(
    "../../gcages/tests/regression/ar6/ar6-workflow-inputs/infilling_db_ar6_cfcs.csv"
)

In [15]:
infiller = AR6Infiller.from_ar6_config(
    ar6_infilling_db_file=AR6_INFILLING_DB_FILE,
    ar6_infilling_db_cfcs_file=AR6_INFILLING_DB_CFCS_FILE,
    n_processes=10,
    # To make sure that our outputs remain harmonised
    # (also, turns out that the historical emissions
    # are the same as the CFCs database)
    historical_emissions=get_ar6_full_historical_emissions(AR6_INFILLING_DB_CFCS_FILE),
    harmonisation_year=harmoniser.harmonisation_year,
)

In [16]:
infilled = infiller(harmonised)
complete_scenarios = pd.concat([harmonised, infilled])
complete_scenarios

9it [00:00, 99.62it/s]
Scenarios to infill:   0%|          | 0/9 [00:00<?, ?it/s]/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/

2015  \
model      scenario                 region variable                unit                           
IMAGE 3.0  EN_INDCi2030_3000f       World  Emissions|BC            Mt BC/yr            9.727424   
                                           Emissions|CH4           Mt CH4/yr         388.072796   
                                           Emissions|CO            Mt CO/yr          934.349885   
                                           Emissions|CO2|Biosphere Mt CO2/yr        3517.440000   
                                           Emissions|CO2|Fossil    Mt CO2/yr       35635.286300   
...                                                                                         ...   
COFFEE 1.1 EN_NPi2020_400f_lowBECCS World  Emissions|C7F16         kt C7F16/yr         0.233800   
                                           Emissions|C8F18         kt C8F18/yr         0.101400   
                                           Emissions|cC4F8         kt cC4F8/yr         1.267200   
                                           Emissions|SO2F2         kt SO2F2/yr         2.531700   
                                           Emissions|HFC245fa      kt HFC245fa/yr      0.000000   

                                                                                           2016  \
model      scenario                 region variable                unit                           
IMAGE 3.0  EN_INDCi2030_3000f       World  Emissions|BC            Mt BC/yr            9.718991   
                                           Emissions|CH4           Mt CH4/yr         386.607263   
                                           Emissions|CO            Mt CO/yr          948.512512   
                                           Emissions|CO2|Biosphere Mt CO2/yr        3831.159324   
                                           Emissions|CO2|Fossil    Mt CO2/yr       36006.593585   
...                                                                                         ...   
COFFEE 1.1 EN_NPi2020_400f_lowBECCS World  Emissions|C7F16         kt C7F16/yr         0.245174   
                                           Emissions|C8F18         kt C8F18/yr         0.106337   
                                           Emissions|cC4F8         kt cC4F8/yr         1.304749   
                                           Emissions|SO2F2         kt SO2F2/yr         2.606677   
                                           Emissions|HFC245fa      kt HFC245fa/yr      0.000000   

                                                                                           2017  \
model      scenario                 region variable                unit                           
IMAGE 3.0  EN_INDCi2030_3000f       World  Emissions|BC            Mt BC/yr            9.710393   
                                           Emissions|CH4           Mt CH4/yr         385.143926   
                                           Emissions|CO            Mt CO/yr          962.644719   
                                           Emissions|CO2|Biosphere Mt CO2/yr        4144.878648   
                                           Emissions|CO2|Fossil    Mt CO2/yr       36377.802621   
...                                                                                         ...   
COFFEE 1.1 EN_NPi2020_400f_lowBECCS World  Emissions|C7F16         kt C7F16/yr         0.256579   
                                           Emissions|C8F18         kt C8F18/yr         0.111288   
                                           Emissions|cC4F8         kt cC4F8/yr         1.342401   
                                           Emissions|SO2F2         kt SO2F2/yr         2.681859   
                                           Emissions|HFC245fa      kt HFC245fa/yr      0.000000   

                                                                                           2018  \
model      scenario                 region variable                unit                           
IMAGE 3.0  EN_INDCi2030_3000f

In [17]:
# Cast infilled dataframe to pyam and add total CO2
df_infilled = pyam.IamDataFrame(complete_scenarios)

# Add total CO2 emissions (Fossil + Biosphere)
df_infilled.add(
    a="Emissions|CO2|Fossil",
    b="Emissions|CO2|Biosphere",
    name="Emissions|CO2",
    append=True
)

df_infilled

<class 'pyam.core.IamDataFrame'>
Index:
 * model    : AIM/CGE 2.2, COFFEE 1.1, GCAM 5.3, IMAGE 3.0, ... WITCH 5.0 (9)
 * scenario : CO_Bridge, DeepElec_SSP2_ HighRE_Budg900, ... SusDev_SDP-PkBudg1000 (9)
Timeseries data coordinates:
   region   : World (1)
   variable : Emissions|BC, Emissions|C2F6, Emissions|C3F8, ... Emissions|cC4F8 (53)
   unit     : Mt BC/yr, Mt CH4/yr, Mt CO/yr, Mt CO2/yr, Mt NH3/yr, ... kt cC4F8/yr (51)
   year     : 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, ... 2100 (86)

In [18]:
# Calculate when CO2 emissions cross zero
calculate_crossing_timings(
    df_infilled,
    variable="Emissions|CO2",
    level=0,
    meta_column="year_netzero_co2"
)

# Display scenarios with their net-zero timings
df_infilled.meta[["year_netzero_co2"]]

,,year_netzero_co2
model,scenario,
AIM/CGE 2.2,EN_NPi2020_900f,2072.0
COFFEE 1.1,EN_NPi2020_400f_lowBECCS,2060.0
GCAM 5.3,NGFS2_Current Policies,NaN
IMAGE 3.0,EN_INDCi2030_3000f,NaN
MESSAGEix-GLOBIOM 1.0,LowEnergyDemand_1.3_IPCC,2052.0
MESSAGEix-GLOBIOM_GEI 1.0,SSP2_openres_lc_50,2062.0
REMIND-MAgPIE 2.1-4.2,SusDev_SDP-PkBudg1000,2067.0
REMIND-MAgPIE 2.1-4.3,DeepElec_SSP2_ HighRE_Budg900,2057.0
WITCH 5.0,CO_Bridge,2076.0


In [19]:
# Create flat-lined variants after net-zero CO2
df_co2_flatline = extend_flatline_after_netzero(
    df_infilled,
    constituent_variables=[
        "Emissions|CO2|Fossil",
        "Emissions|CO2|Biosphere"
    ],
    aggregate_variable="Emissions|CO2",
    netzero_meta_column="year_netzero_co2",
    scenario_suffix="_NZCO2",
)

df_co2_flatline

Processing scenarios: 100%|██████████| 9/9 [00:00<00:00, 108.32it/s]


<class 'pyam.core.IamDataFrame'>
Index:
 * model    : AIM/CGE 2.2, COFFEE 1.1, GCAM 5.3, IMAGE 3.0, ... WITCH 5.0 (9)
 * scenario : CO_Bridge_NZCO2, DeepElec_SSP2_ HighRE_Budg900_NZCO2, ... SusDev_SDP-PkBudg1000_NZCO2 (9)
Timeseries data coordinates:
   region   : World (1)
   variable : Emissions|BC, Emissions|C2F6, Emissions|C3F8, ... Emissions|cC4F8 (53)
   unit     : Mt BC/yr, Mt CH4/yr, Mt CO/yr, Mt CO2/yr, Mt NH3/yr, ... kt cC4F8/yr (51)
   year     : 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, ... 2100 (86)
Meta indicators:
   year_netzero_co2 (float64) 2072.0, 2060.0, nan, 2052.0, 2062.0, 2067.0, ... 2076.0 (8)

create MAGICC flags and run the variants through

In [20]:
MAGICC_EXE_PATH = Path("../../gcages/tests/regression/ar6/ar6-workflow-inputs/magicc-v7.5.3/bin")
MAGICC_AR6_PROBABILISTIC_CONFIG_FILE = Path(
    "../../gcages/tests/regression/ar6/ar6-workflow-inputs/magicc-ar6-0fd0f62-f023edb-drawnset/0fd0f62-derived-metrics-id-f023edb-drawnset.json"
)

In [21]:
if platform.system() == "Darwin":
    if platform.processor() == "arm":
        MAGICC_EXE = MAGICC_EXE_PATH / "magicc-darwin-arm64"
        os.environ["DYLD_LIBRARY_PATH"] = "/opt/homebrew/Cellar/gcc/15.2.0_1/lib/gcc/current"

elif platform.system() == "Linux":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc"

elif platform.system() == "Windows":
    MAGICC_EXE = MAGICC_EXE_PATH / "magicc.exe"

In [22]:
MAGICC_FLAGS = [
    "ALL",
    "GHG",
    "CO2"
]

In [23]:
dfs_to_run = pyam.concat(
    [df_infilled, df_co2_flatline]
)

In [24]:
dfs_to_run.filter(
    variable='Emissions|CO2',
    keep=False,
    inplace=True
)

In [25]:
dfs_to_run_ts = (
    dfs_to_run
    .timeseries()
)

In [26]:
res=[]
for flag in MAGICC_FLAGS:
    scm_runner = AR6SCMRunner.from_ar6_config(
        # Generally, you want to run SCMs in parallel
        n_processes=multiprocessing.cpu_count(),
        magicc_exe_path=MAGICC_EXE,
        magicc_prob_distribution_path=MAGICC_AR6_PROBABILISTIC_CONFIG_FILE,
        historical_emissions=get_ar6_full_historical_emissions(AR6_INFILLING_DB_CFCS_FILE),
        harmonisation_year=2015,
        output_variables=("Surface Air Temperature Change",),
    )
    magicc7_cfgs = [
        {**c, "rf_total_runmodus": flag}
        for c in scm_runner.climate_models_cfgs["MAGICC7"]
    ]
    scm_runner.climate_models_cfgs["MAGICC7"] = magicc7_cfgs
    scm_results = scm_runner(dfs_to_run_ts)
    post_processor = AR6PostProcessor.from_ar6_config(n_processes=10)
    post_processed_results = post_processor(
        scm_results.loc[pix.isin(variable=["Surface Air Temperature Change"])]
    )
    post_processed_results_runid = post_processed_results.timeseries_run_id
    post_processed_results_runid = post_processed_results_runid.assign(
        magicc_runmodus=flag
    )
    post_processed_results_runid.set_index(
        "magicc_runmodus",
        append=True
    )
    res.append(post_processed_results_runid)

Climate models:   0%|          | 0/1 [00:00<?, ?it/s]
[WARNING] 11:08:35 - openscm_runner.adapters.magicc7.magicc7: Historical data has not been checked
/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/scmdata/run.py:2632: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat([ret._meta.to_frame(), *to_join_metas]).astype("category")


Writing SCEN7 files: 100%|██████████| 18.0/18.0 [00:00<00:00, 22.7it/s]


/Users/ganti/Documents/code/ar7_netzero_assessment/.venv/lib/python3.13/site-packages/jwt/api_jwt.py:147: InsecureKeyLengthWarning: The HMAC key is 20 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
Front serial:

In [27]:
res_pyam = pyam.IamDataFrame(
    pd.concat(res)
)

In [28]:
res_pyam

<class 'pyam.core.IamDataFrame'>
Index:
 * model           : AIM/CGE 2.2, COFFEE 1.1, GCAM 5.3, IMAGE 3.0, ... WITCH 5.0 (9)
 * scenario        : CO_Bridge, CO_Bridge_NZCO2, ... SusDev_SDP-PkBudg1000_NZCO2 (18)
Timeseries data coordinates:
   region          : World (1)
   variable        : Surface Temperature (GSAT) (1)
   unit            : K (1)
   year            : 1750, 1751, 1752, 1753, 1754, 1755, 1756, ... 2100 (351)
   climate_model   : MAGICCv7.5.3 (1)
   magicc_runmodus : ALL, CO2, GHG (3)
   run_id          : 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, ... 599 (600)

## Calculate relevant metrics